# Week 7 V2: Rollback-Constrained Unlearning

This notebook runs the rollback experiment in a new result folder. It never overwrites `adaptive_constrained_unlearning_v1`. Rejected quarter-epoch trials restore the last feasible adapter before retrying.

## 1. Install Runtime Dependencies

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes pandas numpy safetensors

## 2. Clone Or Update The GitHub Repo

Add `GITHUB_TOKEN` in Colab Secrets. The sparse checkout includes the Week 3.5 baseline, shared evaluation code, v1 metrics, and v2 files without materializing the large v1 adapter tree.

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

REPOSITORY = "HannanSeyfi/unlearning-thesis"
BRANCH = "main"
FALLBACK_BRANCH = "codex/week7-rollback-constrained-v2"
REPO_DIR = Path("/content/unlearning-thesis")
SPARSE_PATHS = [
    "Tools",
    "Week 3.5/data/synthetic_facts_v1",
    "Week 3.5/data/general_controls_v1",
    "Week 3.5/results/qwen05_high_accuracy_baseline/adapter",
    "Week 3.5/results/reference_successful_run/adapter",
    "Week 4/results/gradient_ascent_unlearning_v1/results",
    "Week 5/results/retain_regularized_unlearning_resumable_v1/results",
    "Week 5/results/aggressive_contrast_evaluation_v1/c09_most_aggressive_epoch_03/results",
    "Week 6/constrained_gradient_unlearning",
    "Week 6/results/constrained_gradient_unlearning_v1/results",
    "Week 7/adaptive_constrained_unlearning",
    "Week 7/results/adaptive_constrained_unlearning_v1/results",
    "Week 7/rollback_constrained_unlearning_v2",
    "Week 7/results/rollback_constrained_unlearning_v2",
]

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, f"https://github.com/{REPOSITORY}.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set", "--cone", *SPARSE_PATHS], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)

%cd {REPO_DIR}
from Tools.github_colab_sync import setup_colab_repo, commit_and_push

THESIS_DIR = setup_colab_repo(REPOSITORY, BRANCH, REPO_DIR)
%cd {THESIS_DIR}

SCRIPT_PATH = THESIS_DIR / "Week 7" / "rollback_constrained_unlearning_v2" / "train_week7_rollback_constrained_unlearning_v2.py"
if not SCRIPT_PATH.exists():
    print(f"V2 script missing on {BRANCH}; checking out {FALLBACK_BRANCH}.")
    subprocess.run(["git", "fetch", "origin", FALLBACK_BRANCH], cwd=THESIS_DIR, check=True)
    subprocess.run(["git", "checkout", "-B", FALLBACK_BRANCH, "FETCH_HEAD"], cwd=THESIS_DIR, check=True)
    BRANCH = FALLBACK_BRANCH
    SCRIPT_PATH = THESIS_DIR / "Week 7" / "rollback_constrained_unlearning_v2" / "train_week7_rollback_constrained_unlearning_v2.py"

print("Using branch:", BRANCH)
print("V2 script:", SCRIPT_PATH)

## 3. Run The Focused Rollback Experiment

Leave `RESET_EXISTING_RUN = False` to resume. The v2 run name and folder are separate from v1.

In [ ]:
RUN_NAME = "rollback_constrained_unlearning_v2"
RESET_EXISTING_RUN = False
MAX_TRIALS = 20
MAX_ACCEPTED_BLOCKS = 16
PUSH_EACH_TRIAL = True

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT_PATH),
    "--repo-root", str(THESIS_DIR),
    "--run-name", RUN_NAME,
    "--max-trials", str(MAX_TRIALS),
    "--max-accepted-blocks", str(MAX_ACCEPTED_BLOCKS),
    "--push-branch", BRANCH,
]
if RESET_EXISTING_RUN:
    cmd.append("--reset")
if PUSH_EACH_TRIAL:
    cmd.append("--push-each-trial")

print("Running:", " ".join(cmd), flush=True)
process = subprocess.Popen(cmd, cwd=THESIS_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
output_tail = deque(maxlen=120)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
    output_tail.append(line)
return_code = process.wait()
if return_code != 0:
    failure_path = THESIS_DIR / "Week 7" / "results" / RUN_NAME / "failure.json"
    if failure_path.exists():
        print("\nSaved v2 failure diagnostics:\n", failure_path.read_text(encoding="utf-8"), flush=True)
    raise RuntimeError(f"Week 7 v2 exited with status {return_code}. Last output:\n{''.join(output_tail)}")

## 4. Inspect The V2 Results

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

V1_DIR = THESIS_DIR / "Week 7" / "results" / "adaptive_constrained_unlearning_v1"
RUN_DIR = THESIS_DIR / "Week 7" / "results" / RUN_NAME
RESULTS_DIR = RUN_DIR / "results"

print("Preserved v1 directory:", V1_DIR)
print("New v2 directory:", RUN_DIR)
display(pd.read_csv(RESULTS_DIR / "week7_v2_cross_week_comparison.csv"))
display(pd.read_csv(RESULTS_DIR / "candidate_final_evaluations.csv"))
display(Markdown((RESULTS_DIR / "week7_v2_rollback_report.md").read_text(encoding="utf-8")))

## 5. Final GitHub Sync

The runner saves every trial. This final call is harmless when nothing remains to commit.

In [ ]:
commit_and_push(
    RUN_DIR,
    "Colab: save Week 7 v2 rollback outputs",
    repo_dir=THESIS_DIR,
    branch=BRANCH,
)